# FinSight-RAG — Phase 2 (Partially Working System)
**A Source-Grounded Question-Answering Assistant for Corporate Financial Filings**  
Lab 9 PSIS · Activity 1: RAG-Based Domain Assistant · Financial Report Analyzer

This notebook reproduces the Phase 2 evidence: load → chunk → embed → Chroma → retrieve → grounded prompt, plus the test suite. Runtime: CPU is sufficient.

## 1. Setup GitHub account

In [1]:
!git clone https://github.com/akashgoyalll/finsight-rag-I025-39-40.git
%cd finsight-rag-I025-39-40
!pip install -q -r requirements.txt
!bash scripts/get_data.sh

Cloning into 'finsight-rag-I025-39-40'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 34 (delta 3), reused 34 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 430.83 KiB | 3.42 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/finsight-rag-I025-39-40
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.7 

## 2. Retrieval evaluation over the 6 test questions (semantic embeddings: all-MiniLM-L6-v2)
No LLM is called. Each question has ground truth verified from the filing; HIT means the retrieved chunks contain the evidence needed to answer.

In [2]:
!EMBEDDINGS=hf python scripts/run_retrieval_demo.py

modules.json: 100% 349/349 [00:00<00:00, 1.39MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 436kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 27.7MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 262kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.77MB/s]

model.safetensors: downloading bytes:  79% 72.2M/90.9M [00:00<00:00, 125MB/s, 5.77MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:00<00:00, 93.4MB/s, 8.19MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:00<00:00, 99.9MB/s, 8.88MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 7025.86it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 975kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 6.54MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 22.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 396kB/s]
config.json: 100% 190/190 [00:00<00:00, 665kB/s]
Embeddings backend : hf
Pages loaded       : 107  (nke-10k-2023.pdf)
Chunks created     : 516  (size=1000

## 3. Offline lexical baseline (TF-IDF) for comparison

In [3]:
!EMBEDDINGS=tfidf python scripts/run_retrieval_demo.py

Embeddings backend : tfidf
Pages loaded       : 107  (nke-10k-2023.pdf)
Chunks created     : 516  (size=1000, overlap=200)
Sample chunk meta  : {'source': 'nke-10k-2023.pdf', 'page': 16, 'total_pages': 107, 'start_index': 3705}
Index build time   : 14.8s   | retriever top-k = 4

Q1 [numeric lookup]
   Q: What were NIKE, Inc.'s total revenues in fiscal 2023?
   truth : $51.2 billion ($51,217 million), +10% reported / +16% currency-neutral
   retrieved pages: [42, 43, 41, 40]   -> MISS (evidence NOT found)
Q2 [factual lookup]
   Q: How many employees did NIKE have as of May 31, 2023?
   truth : approximately 83,700 employees worldwide
   retrieved pages: [28, 9, 79, 9]   -> HIT  (evidence found: 83,700)
Q3 [numeric + explanation]
   Q: How did gross margin change in fiscal 2023 compared to fiscal 2022?
   truth : decreased 250 bps to 43.5% from 46.0%
   retrieved pages: [37, 38, 48, 47]   -> HIT  (evidence found: 250 basis points)
Q4 [numeric lookup]
   Q: How much did NIKE Direct revenu

## 4. Test suite — metadata preservation, retriever, parser, source resolution, LCEL + memory wiring

In [4]:
!python -m pytest tests/ -v -W ignore::DeprecationWarning

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/finsight-rag-I025-39-40
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 6 items                                                              

tests/test_pipeline.py::test_loader_attaches_source_and_page PASSED      [ 16%]
tests/test_pipeline.py::test_metadata_survives_chunking PASSED           [ 33%]
tests/test_pipeline.py::test_retriever_returns_cited_chunks PASSED       [ 50%]
tests/test_pipeline.py::test_parser_rejects_malformed_output PASSED      [ 66%]
tests/test_pipeline.py::test_sources_resolved_from_metadata_not_llm PASSED [ 83%]
tests/test_pipeline.py::test_full_lcel_chain_and_memory_wiring PASSED    [100%]

============================== 6 passed in 56.38s ==============================
